## Interactive map: measurements near selected PFAS users

> [!NOTE]
> Please note that this feature only works properly in the browser. To run the cells, set the variable `RUN_WIDGET` to `True`

Blue = PFAS users (dark: known users / production facilities, light: presumptive). Orange to yellow = `df` measurements within 10 km of the selection, coloured by distance to the nearest selected user.

* **Click** a blue point to add it to the selection or remove it
* **Box / lasso** (map toolbar, top right) replaces the selection with all users inside; *shift* adds to it
* **Double-click** or the *Clear selection* button resets the map

Requires that the `add_distances()` cells above have run (hover text shows the `n_within_*km` counts).

In [ ]:
RUN_WIDGET = True

In [ ]:
import pandas as pd

from pfas_trace.config import DATASET_PRESUMPTIVE_USERS, DATASET_USERS, DATASET_MEASUREMENTS

df_users =  pd.read_parquet(DATASET_USERS)
df_presumptive_users = pd.read_parquet(DATASET_PRESUMPTIVE_USERS)
df_measurements = pd.read_parquet(DATASET_MEASUREMENTS)

In [ ]:
if RUN_WIDGET:
    import ipywidgets as widgets
    import plotly.graph_objects as go
    import numpy as np
    from sklearn.neighbors import BallTree

    R_EARTH_KM = 6371.0088
    DIST_BANDS = [  # (upper bound in km, label, colour)
        (1, "≤ 1 km", "#ff7f0e"),
        (5, "1–5 km", "#ffb366"),
        (10, "5–10 km", "#ffe119"),
    ]
    MEAS_COLS = ["lat", "lon", "name", "matrix", "year", "pfas_sum", "source_type", "dataset_name"]


    def build_measurement_tree(df_meas):
        # same haversine setup as add_distances(), but we keep the row positions so matches can be mapped back to df_meas
        meas_mask = (df_meas["lat"].notna() & df_meas["lon"].notna()).to_numpy()
        meas_pos = np.flatnonzero(meas_mask)
        measurements = np.radians(df_meas.iloc[meas_pos][["lat", "lon"]].to_numpy())
        return BallTree(measurements, metric="haversine"), meas_pos


    def find_measurements_near(df_sel_users, df_meas, tree, meas_pos, max_km=DIST_BANDS[-1][0]):
        """All df_meas rows within max_km of any selected user, with the distance to the nearest selected user."""
        band_labels = [label for _, label, _ in DIST_BANDS]
        if df_sel_users.empty:
            return df_meas.iloc[[]][MEAS_COLS].assign(distance_km=np.array([], dtype=float),
                                                      nearest_user=np.array([], dtype=str),
                                                      band=pd.Categorical([], band_labels))

        users = np.radians(df_sel_users[["lat", "lon"]].to_numpy())
        ind, dist = tree.query_radius(users, r=max_km / R_EARTH_KM, return_distance=True)

        pairs = pd.DataFrame({
            "pos": meas_pos[np.concatenate(ind)],
            "distance_km": np.concatenate(dist) * R_EARTH_KM,
            "nearest_user": np.repeat(df_sel_users["name"].fillna("unnamed").to_numpy(), [len(i) for i in ind]),
        })
        # a measurement can be close to several selected users -> keep only the nearest one
        pairs = pairs.sort_values("distance_km").drop_duplicates("pos")

        return df_meas.iloc[pairs["pos"]][MEAS_COLS].assign(
            distance_km=pairs["distance_km"].to_numpy(),
            nearest_user=pairs["nearest_user"].to_numpy(),
            band=pd.cut(pairs["distance_km"].to_numpy(), bins=[0] + [km for km, _, _ in DIST_BANDS],
                        labels=band_labels, include_lowest=True),
        )


    # ~1s, built once and reused by every map interaction
    meas_tree, meas_pos = build_measurement_tree(df_measurements)
else:
    print("Please run this notebook in the browser and set RUN_WIDGET=True")

In [ ]:
# rerun this cell to reset the map
if not RUN_WIDGET:
    raise ValueError("Please run this notebook in the browser and set RUN_WIDGET=True")

USER_SETS = {  # trace name -> (users, colour)
    "Known PFAS user / production facility": (df_users, "#08306b"),
    "Presumptive PFAS user": (df_presumptive_users, "#6baed6"),
}
users_by_trace = {
    name: users.dropna(subset=["lat", "lon"]).reset_index(names="orig_index")  # point index == row position
    for name, (users, _) in USER_SETS.items()
}


def user_hover(users):
    return (
            "<b>" + users["name"].fillna("unnamed") + "</b><br>" + users["category"] + " · " + users["city"].fillna(
        "?")
            + "<br>within 1 / 5 / 10 km: " + users["n_within_1km"].astype(str) + " / "
            + users["n_within_5km"].astype(str) + " / " + users["n_within_10km"].astype(str)
    )


def measurement_hover(meas):
    # rows with identical coordinates are drawn on top of each other -> summarise the whole location in the hover
    per_year = meas.groupby(["lat", "lon", "year"], dropna=False).size().reset_index(name="n")
    per_year["line"] = per_year["year"].astype("Int64").astype(str).replace("<NA>", "unknown") + ": " + per_year[
        "n"].astype(str)
    location = per_year.groupby(["lat", "lon"]).agg(n_total=("n", "sum"), per_year=("line", "<br>".join))
    meas = meas.join(location, on=["lat", "lon"])
    return (
            "<b>" + meas["name"].fillna("unnamed") + "</b> (" + meas["matrix"].fillna("?") + ", "
            + meas["year"].astype("Int64").astype(str) + ")<br>pfas_sum: " + meas["pfas_sum"].astype(str)
            + "<br>" + meas["distance_km"].round(2).astype(str) + " km from " + meas["nearest_user"]
            + "<br><br><b>" + meas["n_total"].astype(str) + " measurements at this location</b><br>" + meas[
                "per_year"]
    )


fig = go.FigureWidget(
    data=[
        # one trace per distance band so the legend shows the colour grading; filled on selection
        *[go.Scattermap(name=label, lat=[], lon=[], mode="markers", marker=dict(size=7, color=color),
                        unselected=dict(marker=dict(opacity=1)), hovertemplate="%{text}<extra></extra>")
          for _, label, color in DIST_BANDS],
        # users last, so they are drawn on top and stay clickable
        *[go.Scattermap(name=name, lat=users["lat"], lon=users["lon"], text=user_hover(users), mode="markers",
                        marker=dict(size=9, color=USER_SETS[name][1]),
                        selected=dict(marker=dict(size=15)), unselected=dict(marker=dict(opacity=0.35)),
                        hovertemplate="%{text}<extra></extra>")
          for name, users in users_by_trace.items()],
    ],
    layout=dict(
        map=dict(style="carto-positron", center=dict(lat=46.6, lon=2.4), zoom=5),
        height=750, margin=dict(l=0, r=0, t=0, b=0), clickmode="event", hovermode="closest",
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
    ),
)
band_traces = {label: fig.data[i] for i, (_, label, _) in enumerate(DIST_BANDS)}
user_traces = fig.data[len(DIST_BANDS):]

selected = {name: set() for name in users_by_trace}
summary = widgets.HTML()
clear_btn = widgets.Button(description="Clear selection", icon="times")


def refresh():
    df_sel = pd.concat([users_by_trace[name].iloc[sorted(inds)] for name, inds in selected.items()])
    near = find_measurements_near(df_sel, df_measurements, meas_tree, meas_pos)
    nothing_selected = df_sel.empty
    with fig.batch_update():
        for label, trace in band_traces.items():
            band = near[near["band"] == label]
            trace.lat, trace.lon, trace.text = band["lat"], band["lon"], measurement_hover(band)
            trace.selectedpoints = None  # ignore box/lasso selections hitting measurement points
        for trace in user_traces:
            trace.selectedpoints = None if nothing_selected else sorted(selected[trace.name])
    counts = near["band"].value_counts()
    summary.value = (f"<b>{len(df_sel)}</b> users selected · <b>{len(near)}</b> measurements: "
                     + " / ".join(f"<span style='color:{color}'>■</span> {counts[label]} {label}"
                                  for _, label, color in DIST_BANDS))


def make_handler(kind):
    # plotly dispatches every event to every trace (with empty point_inds for unaffected traces), in trace order
    def handler(trace, points, _=None):
        inds = selected[trace.name]
        if kind == "click":
            inds.symmetric_difference_update(points.point_inds)  # click toggles a single user
        else:
            inds.clear()
            inds.update(points.point_inds if kind == "select" else [])
        if trace is user_traces[-1]:  # all user traces have been updated -> recompute once
            refresh()

    return handler


for trace in user_traces:
    trace.on_click(make_handler("click"))
    trace.on_selection(make_handler("select"))
    trace.on_deselect(make_handler("deselect"))


def clear_selection(_):
    for inds in selected.values():
        inds.clear()
    refresh()


clear_btn.on_click(clear_selection)
refresh()
widgets.VBox([widgets.HBox([clear_btn, summary]), fig])